# データ作成

In [1]:
from pathlib import Path

DATA_DIR = Path("/content")

# 厨二病元データ
chuni_files = list(DATA_DIR.glob("*コエノミカタ*はるえるブログ*.txt"))

def load_lines(paths):
    lines = []
    if isinstance(paths, (list, tuple)):
        for p in paths:
            with open(p, encoding="utf-8") as f:
                for line in f:
                    s = line.strip()
                    if s:
                        lines.append(s)
    else:
        with open(paths, encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if s:
                    lines.append(s)
    # 重複除去（順序保持）
    lines = list(dict.fromkeys(lines))
    return lines

chuni = load_lines(chuni_files)

print("厨二病ファイル数:", len(chuni_files))
print("厨二病文数:", len(chuni))
print("厨二病サンプル:", chuni[0])

# 厨二病コーパスとして保存
with open("chuni_sentences.txt", "w", encoding="utf-8") as f:
    for s in chuni:
        f.write(s + "\n")
print("\nchuni_sentences.txt を保存しました")

厨二病ファイル数: 6
厨二病文数: 130
厨二病サンプル: 妖精さんたち！私に力を貸して！！！魅惑の香りに飲み込まれなさい"ポイズン・ベリー"！！！

chuni_sentences.txt を保存しました


In [2]:
import requests, time, re

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "ChuniVector/1.0 (educational; contact:none)"
})

def get_random_wikipedia_intro(timeout=10):
    url = "https://ja.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "random",
        "rnnamespace": 0,   # 記事
        "rnlimit": 1,
        "format": "json"
    }
    try:
        r = SESSION.get(url, params=params, timeout=timeout)
        if r.status_code != 200:
            return ""
        title = r.json()["query"]["random"][0]["title"]
    except Exception:
        return ""

    # intro取得
    params2 = {
        "action": "query",
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "format": "json",
        "formatversion": 2,
        "titles": title
    }
    try:
        r2 = SESSION.get(url, params=params2, timeout=timeout)
        if r2.status_code != 200:
            return ""
        pages = r2.json().get("query", {}).get("pages", [])
        if not pages:
            return ""
        return pages[0].get("extract", "") or ""
    except Exception:
        return ""

def split_sentences(text):
    # 「。」が少ない記事もあるので，改行や！？も分割対象にする
    text = text.replace("\n", " ")
    parts = re.split(r"[。！？]", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

target = 130
normal_sentences = []
seen = set()
tries = 0

while len(normal_sentences) < target and tries < 800:
    tries += 1
    intro = get_random_wikipedia_intro()
    if not intro:
        time.sleep(0.2)
        continue

    for s in split_sentences(intro):
        if len(s) < 15:
            continue
        if s in seen:
            continue
        seen.add(s)
        normal_sentences.append(s)
        if len(normal_sentences) >= target:
            break

    time.sleep(0.2)  # 429対策

print(f"普通文数: {len(normal_sentences)} / 目標 {target}（試行 {tries}）")
print("サンプル:")
for s in normal_sentences[:5]:
    print("-", s)

with open("normal_sentences.txt", "w", encoding="utf-8") as f:
    for s in normal_sentences:
        f.write(s + "\n")
print("\nnormal_sentences.txt を保存しました")

普通文数: 130 / 目標 130（試行 53）
サンプル:
- 『ラム』（英語: Ram）は、1971年に発表された“ポール・マッカートニー&リンダ・マッカートニー”によるアルバム
- 全英で2週1位、全米2位を記録
- 圓福寺（えんふくじ）は、宮崎県高鍋町南高鍋にある浄土宗寺院
- オオバンブルマイ（欧字名:Obamburumai 香:大宴重酬、2020年2月26日 - ）は、日本の競走馬
- 主な勝ち鞍は2022年の京王杯2歳ステークス、2023年のアーリントンカップ、ゴールデンイーグル

normal_sentences.txt を保存しました
